## Plotting AUC against Trials and saving as PDF

In [0]:
import plotly.graph_objects as go

# Trial data with model types
trials = [
    (1,  0.6525, "LR"),  (2,  0.6251, "RF"),  (3,  0.5383, "GBT"),
    (4,  0.9115, "LR"),  (5,  0.9139, "LR"),  (6,  0.9147, "GBT"),
    (7,  0.9114, "LR"),  (8,  0.9005, "LR"),  (9,  0.9237, "LR"),
    (10, 0.9350, "LR"),  (11, 0.9411, "RF"),  (12, 0.9156, "GBT"),
    (13, 0.9240, "RF"),  (14, 0.9360, "GBT"), (15, 0.9379, "MLP"),
    (16, 0.9238, "MLP"), (17, 0.9136, "MLP"), (18, 0.9137, "MLP"),
    (19, 0.9034, "MLP"), (20, 0.8837, "MLP"), (21, 0.8847, "MLP"),
    (22, 0.9314, "MLP"), (23, 0.9452, "MLP"), (24, 0.9121, "MLP"),
    (25, 0.8994, "MLP"), (26, 0.9235, "MLP"), (27, 0.9227, "MLP"),
    (28, 0.8610, "MLP"), (29, 0.8817, "MLP"), (30, 0.8993, "MLP"), (31, 0.9304, "MLP"),
]

trial_nums = [t[0] for t in trials]
aucs = [t[1] for t in trials]
model_types = [t[2] for t in trials]

colors = {"LR": "#8bbcc4", "RF": "#f2ccda", "GBT": "#e8b898", "MLP": "#a68ec2"}
full_names = {"LR": "Logistic Regression", "RF": "Random Forest", "GBT": "Gradient Boosted Trees", "MLP": "Multilayer Perceptron"}

fig = go.Figure()

# Connecting line
fig.add_trace(go.Scatter(
    x=trial_nums, y=aucs, mode="lines",
    line=dict(color="#8e9699", width=1.5),
    showlegend=False, hoverinfo="skip"
))

# Points colored by model type
for mt in ["LR", "RF", "GBT", "MLP"]:
    idx = [i for i, m in enumerate(model_types) if m == mt]
    fig.add_trace(go.Scatter(
        x=[trial_nums[i] for i in idx],
        y=[aucs[i] for i in idx],
        mode="markers",
        name=full_names[mt],
        marker=dict(color=colors[mt], size=10, line=dict(width=1, color="white")),
        hovertemplate="Trial %{x}<br>AUC: %{y:.4f}<extra>" + full_names[mt] + "</extra>"
    ))

# Highlight last trial (Trial 32)
best_idx = len(aucs) - 1
fig.add_annotation(
    x=trial_nums[best_idx], y=aucs[best_idx],
    text=f"<b>Best: {aucs[best_idx]:.4f}</b>",
    showarrow=True, arrowhead=2, arrowcolor="#333",
    ax=30, ay=-30,
    font=dict(size=10, color="#333"),
    bgcolor="#fff8ad", bordercolor="#333", borderwidth=1
)

# Phase annotations
fig.add_vrect(x0=0.5, x1=3.5, fillcolor="#f0f0f0", opacity=0.3, line_width=0,
              annotation_text="Initial\nbaseline", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")
fig.add_vrect(x0=3.5, x1=14.5, fillcolor="#e8f4fd", opacity=0.2, line_width=0,
              annotation_text="Feature engineering\n& tuning", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")
fig.add_vrect(x0=14.5, x1=23.5, fillcolor="#f3e8fd", opacity=0.2, line_width=0,
              annotation_text="MLP\nexploration", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")
fig.add_vrect(x0=23.5, x1=32.5, fillcolor="#e8fdfb", opacity=0.2, line_width=0,
              annotation_text="OHE\nvector\ndropped", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")

fig.update_layout(
    title=dict(text="<b>Model Development: AUC Across Trials</b>", font=dict(size=16), x=0.5, xanchor="center", y=0.95, yanchor="top"),
    xaxis=dict(
        title=dict(text="<b>Trial</b>", standoff=10),
        showgrid=True, gridcolor="#eeeeee", dtick=1, range=[0, 33]
    ),
    yaxis=dict(
        title=dict(text="<b>AUC</b>"),
        showgrid=True, gridcolor="#eeeeee", range=[0.5, 1.0]
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    height=450
)
fig.show()

print(f"Best AUC: {aucs[best_idx]:.4f} (Trial {trial_nums[best_idx]}, {full_names[model_types[best_idx]]})")
print(f"AUC improvement from baseline: {aucs[best_idx] - aucs[0]:.4f} (+{((aucs[best_idx]/aucs[0])-1)*100:.1f}%)")

In [0]:
import subprocess, sys, importlib, base64

# Install kaleido to temp dir
target_dir = "/tmp/kaleido_pkg"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--target", target_dir, "kaleido==0.2.1"])
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)
importlib.invalidate_caches()

# Use kaleido directly — disable MathJax by bypassing the default setter
from kaleido.scopes.plotly import PlotlyScope
scope = PlotlyScope()
scope._mathjax = None  # Forces --mathjax flag to NOT be passed to chromium

# Crop the top by reducing top margin in the exported figure
import copy
fig_export = copy.deepcopy(fig.to_dict())
fig_export["layout"]["margin"] = {"t": 50, "b": 50, "l": 60, "r": 50}

# Export figure as PDF
pdf_bytes = scope.transform(
    fig_export, format="pdf", width=1000, height=450
)
print(f"PDF generated: {len(pdf_bytes)} bytes")

# Download link
b64 = base64.b64encode(pdf_bytes).decode()
displayHTML(f'<a href="data:application/pdf;base64,{b64}" download="AUC_Across_Trials.pdf" style="font-size:16px; padding:12px 24px; background:#4CAF50; color:white; text-decoration:none; border-radius:5px;">\U0001f4e5 Download PDF for LaTeX</a>')

In [0]:
import copy, base64

TEXT_COLOR = "#f0ede6"

# Create export copy with transparent background
fig_png = copy.deepcopy(fig.to_dict())
fig_png["layout"]["paper_bgcolor"] = "rgba(0,0,0,0)"
fig_png["layout"]["plot_bgcolor"] = "rgba(0,0,0,0)"
fig_png["layout"]["margin"] = {"t": 50, "b": 50, "l": 60, "r": 50}

# Remove grid
fig_png["layout"]["xaxis"]["showgrid"] = False
fig_png["layout"]["yaxis"]["showgrid"] = False

# Axis lines in TEXT_COLOR
fig_png["layout"]["xaxis"]["showline"] = True
fig_png["layout"]["xaxis"]["linecolor"] = TEXT_COLOR
fig_png["layout"]["xaxis"]["linewidth"] = 1
fig_png["layout"]["xaxis"]["tickfont"] = {"color": TEXT_COLOR}
fig_png["layout"]["xaxis"]["title"]["font"] = {"color": TEXT_COLOR}
fig_png["layout"]["yaxis"]["showline"] = True
fig_png["layout"]["yaxis"]["linecolor"] = TEXT_COLOR
fig_png["layout"]["yaxis"]["linewidth"] = 1
fig_png["layout"]["yaxis"]["tickfont"] = {"color": TEXT_COLOR}
fig_png["layout"]["yaxis"]["title"]["font"] = {"color": TEXT_COLOR}

# Title text
fig_png["layout"]["title"]["font"]["color"] = TEXT_COLOR

# Legend text
fig_png["layout"]["legend"]["font"] = {"color": TEXT_COLOR}

# Remove section background colors (vrects) and replace with dashed lines
fig_png["layout"]["shapes"] = [
    {"type": "line", "x0": 3.5, "x1": 3.5, "y0": 0, "y1": 1, "yref": "paper",
     "line": {"color": TEXT_COLOR, "width": 1, "dash": "dash"}},
    {"type": "line", "x0": 14.5, "x1": 14.5, "y0": 0, "y1": 1, "yref": "paper",
     "line": {"color": TEXT_COLOR, "width": 1, "dash": "dash"}},
    {"type": "line", "x0": 23.5, "x1": 23.5, "y0": 0, "y1": 1, "yref": "paper",
     "line": {"color": TEXT_COLOR, "width": 1, "dash": "dash"}},
]

# Change connecting line color to TEXT_COLOR
for trace in fig_png["data"]:
    if trace.get("mode") == "lines":
        trace["line"]["color"] = TEXT_COLOR
    # Remove dot outlines
    if trace.get("mode") == "markers" and "marker" in trace:
        trace["marker"]["line"] = {"width": 0}

# All annotations text in TEXT_COLOR, remove background styling,
# and shift section labels to the right (except "Initial baseline")
for ann in fig_png["layout"].get("annotations", []):
    if "font" in ann:
        ann["font"]["color"] = TEXT_COLOR
    else:
        ann["font"] = {"color": TEXT_COLOR}
    ann.pop("bgcolor", None)
    ann.pop("bordercolor", None)
    ann.pop("borderwidth", None)
    # Change arrow color to TEXT_COLOR
    if "arrowcolor" in ann:
        ann["arrowcolor"] = TEXT_COLOR
    # Shift non-initial section labels a bit to the right
    text = ann.get("text", "")
    if "Initial" not in text and "Best" not in text and "x" in ann:
        ann["x"] = ann["x"] + 0.8

# Export as PNG
png_bytes = scope.transform(
    fig_png, format="png", width=1000, height=450, scale=2
)
print(f"PNG generated: {len(png_bytes)} bytes")

# Download link
b64 = base64.b64encode(png_bytes).decode()
displayHTML(f'<a href="data:image/png;base64,{b64}" download="AUC_Across_Trials.png" style="font-size:16px; padding:12px 24px; background:#4CAF50; color:white; text-decoration:none; border-radius:5px;">\U0001f4e5 Download PNG (transparent)</a>')

## Global SHAP Value calculations:

In [0]:
%pip install shap -q

In [0]:
import shap
import numpy as np
import mlflow
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors
from pyspark.ml.functions import vector_to_array

# Match Plotly default font (same as AUC chart)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Open Sans', 'Arial', 'Helvetica', 'DejaVu Sans']

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# --- Load latest model (v7: 112 features) ---
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/7")

# --- Load training data (112 features, 15-day horizon) ---
spark.catalog.refreshTable("teams.sensorx.train_data")
train = spark.read.table("teams.sensorx.train_data")

# --- Feature names: hardcoded from assembler order (NO onTime) ---
# sensor_cols + lag_cols + dev_cols + ["deviceId_index"] = 112
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_cols = [
    "payload_xrayController_filamentCurrent_avg_daily",
    "payload_xrayController_filamentCurrent_dev_daily",
    "payload_xrayController_temperature_avg_daily",
    "payload_xrayController_temperature_dev_daily",
    "payload_xrayController_tubeCurrent_avg_daily",
    "payload_xrayController_tubeCurrent_dev_daily",
]
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
short_names = [n.replace("payload_xrayController_", "").replace("_avg_daily", "_avg").replace("_dev_daily", "_dev") for n in feature_names]

# Verify
actual_size = train.select(vector_to_array("features").alias("a")).limit(1).collect()[0]["a"]
assert len(actual_size) == len(feature_names) == 112, f"Mismatch: vector={len(actual_size)}, names={len(feature_names)}"
print(f"Features: {len(feature_names)} (no onTime) | First 5: {short_names[:5]}")

# --- Sample training data to numpy ---
N_BG = 500       # background samples
N_EXPLAIN = 500  # samples to explain
N_SAMPLES = 200  # SHAP perturbations

X = np.array(
    train.select(vector_to_array("features").alias("a"))
    .orderBy(F.rand(seed=42)).limit(N_BG + N_EXPLAIN)
    .toPandas()["a"].tolist()
)
background = shap.kmeans(X[:N_BG], 50)
X_explain = X[N_BG:]

# --- Prediction wrapper ---
def predict(X):
    rows = [Row(features=Vectors.dense(x.tolist())) for x in X]
    df = spark.createDataFrame(rows)
    probs = model.transform(df).select(vector_to_array("probability").alias("p")).toPandas()
    return np.array(probs["p"].tolist())

# --- Compute SHAP ---
print(f"Computing SHAP (N_BG={N_BG}, N_EXPLAIN={N_EXPLAIN}, N_SAMPLES={N_SAMPLES})...")
print("This will take a while with increased samples...")
explainer = shap.KernelExplainer(predict, background)
shap_values = explainer.shap_values(X_explain, nsamples=N_SAMPLES, silent=True)

# Extract class 1 (fault) SHAP values
if isinstance(shap_values, list):
    sv = shap_values[1]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]
else:
    sv = shap_values

# --- Top 10 features ---
mean_abs = np.abs(sv).mean(axis=0)
top_idx = np.argsort(mean_abs)[::-1][:10]
print("\nTop 10 global features (MLP v7, 15-day, no onTime):")
for i, idx in enumerate(top_idx, 1):
    print(f"  {i}. {short_names[idx]}: {mean_abs[idx]:.4f}")

# --- Custom colormap ---
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_shap", ["#8bbcc4", "#f2ccda", "#e8b898", "#a68ec2"]
)

# --- SHAP summary plot (white background) ---
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_explain, feature_names=short_names, max_display=10, show=False, cmap=custom_cmap)
plt.gcf().set_facecolor("white")
plt.gca().set_facecolor("white")
plt.title("MLP v7 (15-day) \u2014 Global SHAP (112 features)", fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import base64
from io import BytesIO
import shap

# Match Plotly default font (same as AUC chart)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Open Sans', 'Arial', 'Helvetica', 'DejaVu Sans']

# Recreate SHAP summary plot with custom colors and save to PDF
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_explain, feature_names=short_names, max_display=10, show=False, cmap=custom_cmap)
plt.gcf().set_facecolor("white")
plt.gca().set_facecolor("white")
plt.title("MLP v7 (15-day) \u2014 Global SHAP (112 features)", fontweight="bold")
plt.tight_layout()

# Save to PDF
pdf_buffer = BytesIO()
plt.savefig(pdf_buffer, format='pdf', bbox_inches='tight', dpi=150)
pdf_buffer.seek(0)
pdf_bytes = pdf_buffer.read()
plt.close()

print(f"PDF generated: {len(pdf_bytes)} bytes")

# Download link
b64 = base64.b64encode(pdf_bytes).decode()
displayHTML(f'<a href="data:application/pdf;base64,{b64}" download="SHAP_Summary_MLP_v7.pdf" style="font-size:16px; padding:12px 24px; background:#4CAF50; color:white; text-decoration:none; border-radius:5px;">\U0001f4e5 Download SHAP PDF for LaTeX</a>')

In [0]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import base64
from io import BytesIO
import shap

# Match Plotly default font (same as AUC chart)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Open Sans', 'Arial', 'Helvetica', 'DejaVu Sans']

TEXT_COLOR = "#f0ede6"

# Recreate SHAP summary plot with transparent background
fig_png = plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_explain, feature_names=short_names, max_display=10, show=False, cmap=custom_cmap)
plt.gcf().set_facecolor("none")
plt.gca().set_facecolor("none")
plt.title("MLP v7 (15-day) \u2014 Global SHAP (112 features)", fontweight="bold", color=TEXT_COLOR)

# Set ALL text elements to #f0ede6
for ax in plt.gcf().axes:
    ax.title.set_color(TEXT_COLOR)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.tick_params(axis="both", colors=TEXT_COLOR)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_color(TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(TEXT_COLOR)

# Catch any figure-level text objects (e.g., SHAP "Low"/"High" labels)
for text in plt.gcf().texts:
    text.set_color(TEXT_COLOR)

plt.tight_layout()

# Save to PNG with transparent background
png_buffer = BytesIO()
plt.savefig(png_buffer, format='png', bbox_inches='tight', dpi=200, transparent=True)
png_buffer.seek(0)
png_bytes = png_buffer.read()
plt.close()

print(f"PNG generated: {len(png_bytes)} bytes")

# Download link
b64 = base64.b64encode(png_bytes).decode()
displayHTML(f'<a href="data:image/png;base64,{b64}" download="SHAP_Summary_MLP_v7.png" style="font-size:16px; padding:12px 24px; background:#4CAF50; color:white; text-decoration:none; border-radius:5px;">\U0001f4e5 Download SHAP PNG (transparent)</a>')

In [0]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import base64
from io import BytesIO
import shap

# Match Plotly default font (same as AUC chart)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Open Sans', 'Arial', 'Helvetica', 'DejaVu Sans']

TEXT_COLOR = "#f0ede6"

# Recreate SHAP summary plot with transparent background
fig_png = plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_explain, feature_names=short_names, max_display=10, show=False, cmap=custom_cmap)
plt.gcf().set_facecolor("none")
plt.gca().set_facecolor("none")
plt.title("MLP v7 (15-day) \u2014 Global SHAP (112 features)", fontweight="bold", color=TEXT_COLOR)

# Set ALL text elements to #f0ede6
for ax in plt.gcf().axes:
    ax.title.set_color(TEXT_COLOR)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.tick_params(axis="both", colors=TEXT_COLOR)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_color(TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(TEXT_COLOR)

# Catch any figure-level text objects (e.g., SHAP "Low"/"High" labels)
for text in plt.gcf().texts:
    text.set_color(TEXT_COLOR)

plt.tight_layout()

# Save to PNG with transparent background
png_buffer = BytesIO()
plt.savefig(png_buffer, format='png', bbox_inches='tight', dpi=200, transparent=True)
png_buffer.seek(0)
png_bytes = png_buffer.read()
plt.close()

print(f"PNG generated: {len(png_bytes)} bytes")

# Download link
b64 = base64.b64encode(png_bytes).decode()
displayHTML(f'<a href="data:image/png;base64,{b64}" download="SHAP_Summary_MLP_v7.png" style="font-size:16px; padding:12px 24px; background:#4CAF50; color:white; text-decoration:none; border-radius:5px;">\U0001f4e5 Download SHAP PNG (transparent)</a>')